Импорты

In [1]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
from datetime import datetime,  timezone

Пути и дата, с которой берем посты

In [2]:
data_path = "../data/channels_data/raw/"
output_path = "../data/channels_data/interim/"
stats_path = "../data/stats/"

Path(output_path).mkdir(parents=True, exist_ok=True)
Path(stats_path).mkdir(parents=True, exist_ok=True)

csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]

# берем данные с 1 декабря 2024
start_date = datetime(2024, 12, 1, tzinfo=timezone.utc)

Фильтрация по дате, удаление неиспользованных реакций, статистика по кастомным реакциям

In [3]:
all_stats = {}

for csv_file in csv_files:
    df = pd.read_csv(os.path.join(data_path, csv_file))

    if 'post_datetime' in df.columns:
        date_column = 'post_datetime'
        df['post_datetime'] = pd.to_datetime(df['post_datetime'])
    elif 'post_date' in df.columns and 'post_time' in df.columns:
        date_column = 'post_datetime'
        df['post_datetime'] = pd.to_datetime(df['post_date'] + ' ' + df['post_time'])
    elif 'post_date' in df.columns:
        date_column = 'post_date'
        df['post_date'] = pd.to_datetime(df['post_date'])
    else:
        print(f"{csv_file}: нет столбца с датой, пропускаем")
        continue

    df_filtered = df[df[date_column] >= start_date].copy()
    reaction_columns = [c for c in df_filtered.columns if c.startswith('reaction_')]
    file_stats = {
        'filename': csv_file,
        'total_rows': len(df),
        'filtered_rows': len(df_filtered),
        'filtered_percentage': len(df_filtered) / len(df) * 100 if len(df) else 0,
        'total_reactions': len(reaction_columns),
    }

    if len(df_filtered) == 0 or not reaction_columns:
        df_filtered.to_csv(os.path.join(output_path, csv_file), index=False)
        all_stats[csv_file] = file_stats
        print(f"{csv_file}: {len(df_filtered)} из {len(df)} постов, реакций нет")
        continue

    reaction_stats = []
    for col in reaction_columns:
        series = df_filtered[col].fillna(0)
        total = series.sum()
        posts_with = (series > 0).sum()
        stat = {
            'reaction': col,
            'total_count': int(total),
            'posts_with_reaction': int(posts_with),
            'percentage_of_posts': round(posts_with / len(df_filtered) * 100, 2),
            'status': 'kept' if total > 0 else 'removed',
        }
        if total > 0:
            # первый пост, где реакция встретилась
            idx = (series > 0).idxmax()
            first = df_filtered.loc[idx]
            stat['example_post'] = {
                'post_id': str(first.get('post_id')),
                'post_datetime': str(first.get('post_datetime')),
                'post_date': str(first.get('post_date')),
                'post_time': str(first.get('post_time')),
                'reaction_count': int(series.loc[idx]),
            }
        reaction_stats.append(stat)

    # реакции, которых не было ни разу, удаляем
    kept = [s['reaction'] for s in reaction_stats if s['status'] == 'kept']
    reaction_stats.sort(key=lambda x: x['total_count'], reverse=True)
    file_stats.update(
        kept_reactions=len(kept),
        removed_reactions=len(reaction_columns) - len(kept),
        reaction_details=reaction_stats,
    )

    non_reaction_columns = [c for c in df_filtered.columns if not c.startswith('reaction_')]
    df_filtered[non_reaction_columns + kept].to_csv(os.path.join(output_path, csv_file), index=False)
    all_stats[csv_file] = file_stats
    print(f"{csv_file}: {len(df_filtered)} из {len(df)} постов, реакций {len(kept)} из {len(reaction_columns)}")

with open(os.path.join(stats_path, 'processing_stats.json'), 'w', encoding='utf-8') as f:
    json.dump(all_stats, f, ensure_ascii=False, indent=2, default=str)

# кастомные реакции: в названии столбца есть цифры (id)
custom_rows = []
for csv_file, file_stats in all_stats.items():
    for r in file_stats.get('reaction_details', []):
        if r['status'] == 'kept' and any(ch.isdigit() for ch in r['reaction']):
            ex = r['example_post']
            custom_rows.append({
                'channel': csv_file,
                'reaction_name': r['reaction'],
                'total_uses': r['total_count'],
                'posts_with_reaction': r['posts_with_reaction'],
                'percentage_of_posts': r['percentage_of_posts'],
                'example_post_id': ex['post_id'],
                'example_post_date': ex['post_date'],
                'example_post_time': ex['post_time'],
                'example_post_datetime': ex['post_datetime'],
                'reaction_count_in_post': ex['reaction_count'],
                'status': r['status'],
            })

if custom_rows:
    custom_df = pd.DataFrame(custom_rows)
    custom_df.to_csv(os.path.join(stats_path, 'custom_reactions_detailed.csv'), index=False, encoding='utf-8')
    print(custom_df[['channel', 'reaction_name', 'total_uses', 'example_post_date', 'example_post_id']].to_string(index=False))
else:
    print("Кастомных реакций не найдено")

AI_point_of_view.csv: 394 из 787 постов, реакций 33 из 44
DSCSproAI.csv: 370 из 731 постов, реакций 53 из 77
d_code.csv: 3223 из 9582 постов, реакций 10 из 31
Education_Yandex.csv: 415 из 1381 постов, реакций 24 из 28
ict_moscow_ai.csv: 693 из 4354 постов, реакций 26 из 27
naebnet.csv: 2493 из 6599 постов, реакций 10 из 13
seeallochnaya.csv: 815 из 2553 постов, реакций 17 из 49
skoltech_daily.csv: 872 из 3683 постов, реакций 24 из 42
spbuniversity1724.csv: 1169 из 3166 постов, реакций 33 из 57
tbank_education.csv: 319 из 1436 постов, реакций 9 из 16
              channel                       reaction_name  total_uses example_post_date example_post_id
        DSCSproAI.csv reaction_custom_5368748388885999718         287        2025-04-04             776
        DSCSproAI.csv reaction_custom_5307602562990492983         261        2025-04-08             788
        DSCSproAI.csv reaction_custom_5357314052072692864         180        2025-04-07             785
        DSCSproAI.csv reacti

Замена кастомных реакций на обычные эмодзи по карте каналов и объединение всех вариантов сердечек в одно (interim -> processed)

In [4]:
import pandas as pd
import os
from pathlib import Path

# id кастомной реакции -> обычный эмодзи (здесь все сердечки сразу идут в красное)
CUSTOM_REACTIONS_MAP = {
    'by_channel': {
        'ict_moscow_ai': {
            '5217962652742996135': '❤️',
            '5289509472090220232': '❤️',
            '5445042860787252192': '❤️',
        },
        'd_code': {
            '5348351516182857544': '❤️',
            '5350663853560581569': '❤️',
        },
        'DSCSproAI': {
            '5436189026224728788': '🎉',
            '5438237047020088735': '🤖',
            '5235807206770750407': '😎',
            '5458801195814508191': '🤩',
            '5435931719028997627': '❤️',
            '5368748388885999718': '❤️',
            '4960832663062578456': '❤️',
            '5307602562990492983': '🔥',
            '5472411062412254753': '🧑‍🎓',
            '5435968389459767071': '🔥',
            '5436383132976702162': '⭐',
            '5438544137181752811': '⚡',
            '4961156112754672853': '⚡',
            '5323772371830588991': '🫡',
            '5361918153934791430': '❤️',
            '5424815192615706042': '❤️',
            '5818845400840277279': '👀',
            '4961121357879313223': '❤️',
            '6032706485526465658': '❤️',
            '5235826817591420001': '👨‍💻',
            '5395732581780040886': '🤝',
        },
        'seeallochnaya': {
            '5355083340548419513': '🥹',
            '5248951709367017473': '🚫',
        },
        'spbuniversity1724': {
            '5422748398518301870': '❤️',
            '5420436396148027032': '❤️',
            '5411468637577965032': '❤️',
            '5307926626862909975': '❤️',
            '5388584785536904914': '❤️',
            '5388802501724092155': '❤️',
        },
        'naebnet': {
            '5235478122081560535': '👍',
        }
    }
}

# варианты сердечек, которые сливаем в reaction_❤️
HEART_VARIATIONS = [
    '❤️',
    '🩷',
    '💘',
    '❤‍🔥',
    '❤',
]

ALL_CHANNELS = [
    'Education_Yandex',
    'skoltech_daily',
    'spbuniversity1724',
    'AI_point_of_view',
    'seeallochnaya',
    'tbank_education',
    'd_code',
    'naebnet',
    'ict_moscow_ai',
    'DSCSproAI'
]

input_path = "../data/channels_data/interim/"
output_path = "../data/channels_data/processed/"
Path(output_path).mkdir(parents=True, exist_ok=True)


def get_channel_name_from_filename(filename):
    base_name = filename.replace('.csv', '').lower()
    for channel_name in ALL_CHANNELS:
        if channel_name.lower() in base_name:
            return channel_name
    for channel_name in ALL_CHANNELS:
        channel_simple = channel_name.replace('_', '').lower()
        base_simple = base_name.replace('_', '')
        if channel_simple in base_simple or base_simple in channel_simple:
            return channel_name
    return filename.replace('.csv', '')


def standardize_hearts(df):
    """Складывает все варианты сердечек в один столбец reaction_❤️"""
    df = df.copy()
    standard = 'reaction_❤️'
    # у вариантов общий префикс, поэтому без dict.fromkeys столбцы дублировались бы
    heart_columns = list(dict.fromkeys(
        c for h in HEART_VARIATIONS for c in df.columns if c.startswith(f'reaction_{h}')
    ))
    to_merge = [c for c in heart_columns if c != standard]
    if standard not in df.columns:
        df[standard] = 0
    for col in to_merge:
        df[standard] = df[standard].fillna(0) + df[col].fillna(0)
    return df.drop(columns=to_merge)


def process_custom_reactions(df, channel_name):
    """Складывает кастомные реакции со столбцами обычных эмодзи по карте канала"""
    df = df.copy()
    for custom_id, emoji in CUSTOM_REACTIONS_MAP['by_channel'].get(channel_name, {}).items():
        candidates = (f"reaction_custom_{custom_id}", f"reaction_{custom_id}", custom_id)
        custom_column = next((c for c in candidates if c in df.columns), None)
        if custom_column is None:
            continue

        emoji_column = f"reaction_{emoji}"
        if emoji_column in df.columns:
            df[emoji_column] = df[emoji_column].fillna(0) + df[custom_column].fillna(0)
        else:
            df[emoji_column] = df[custom_column].fillna(0)
        df = df.drop(columns=[custom_column])
    return df


all_stats = {}
for csv_file in sorted(f for f in os.listdir(input_path) if f.endswith('.csv')):
    channel_name = get_channel_name_from_filename(csv_file)
    df = pd.read_csv(os.path.join(input_path, csv_file))
    n_before = len([c for c in df.columns if c.startswith('reaction_')])

    df = process_custom_reactions(standardize_hearts(df), channel_name)
    df.to_csv(os.path.join(output_path, csv_file), index=False)

    n_after = len([c for c in df.columns if c.startswith('reaction_')])
    all_stats[csv_file] = {'channel': channel_name, 'reactions_before': n_before, 'reactions_after': n_after}

pd.DataFrame(all_stats).T

,channel,reactions_before,reactions_after
AI_point_of_view.csv,AI_point_of_view,33,31
DSCSproAI.csv,DSCSproAI,53,40
Education_Yandex.csv,Education_Yandex,24,23
d_code.csv,d_code,10,10
ict_moscow_ai.csv,ict_moscow_ai,26,26
naebnet.csv,naebnet,10,9
seeallochnaya.csv,seeallochnaya,17,17
skoltech_daily.csv,skoltech_daily,24,24
spbuniversity1724.csv,spbuniversity1724,33,26
tbank_education.csv,tbank_education,9,9
